# 🌸 Checkpoint 5 — Entraîner et Évaluer un Modèle (Iris)
## Module Data Science — Machine Learning

---

Projet complet de bout en bout sur le dataset Iris (intégré à Scikit-learn, aucun téléchargement).

### Étapes
1. Charger et explorer
2. Visualiser
3. Préparer (split + normalisation)
4. Entraîner 3 modèles
5. Évaluer
6. Optimiser et comparer


In [ ]:
!pip install scikit-learn seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

sns.set_theme(style="whitegrid")
print("✅ Prêt")

---
## 1. Charger et explorer

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = [iris.target_names[i] for i in iris.target]

print("Dimensions :", df.shape)
print(df["species"].value_counts())
df.head()

In [ ]:
df.describe()

---
## 2. Visualiser

In [ ]:
sns.pairplot(df, hue="species", height=2)
plt.suptitle("Relations entre les mesures des iris", y=1.02)
plt.show()
# → Setosa bien isolée ; Versicolor et Virginica se chevauchent

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop(columns=["species"]).corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Corrélation entre les mesures")
plt.show()

---
## 3. Préparer les données

In [ ]:
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")

---
## 4. Entraîner les 3 modèles

In [ ]:
# Arbre (pas de normalisation nécessaire)
arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

# KNN (données normalisées)
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)

# Régression logistique (classification, données normalisées)
logreg = LogisticRegression(max_iter=200).fit(X_train_s, y_train)

print("✅ 3 modèles entraînés")

---
## 5. Évaluer

In [ ]:
resultats = {
    "Arbre de décision"     : accuracy_score(y_test, arbre.predict(X_test)),
    "KNN (k=5)"             : accuracy_score(y_test, knn.predict(X_test_s)),
    "Régression logistique" : accuracy_score(y_test, logreg.predict(X_test_s)),
}
for m, s in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
    print(f"{m:25} : {s:.3f}")

In [ ]:
# Matrice de confusion (arbre)
cm = confusion_matrix(y_test, arbre.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot(cmap="Blues")
plt.title("Matrice de confusion — Arbre")
plt.show()

In [ ]:
# Rapport de classification détaillé
print(classification_report(y_test, arbre.predict(X_test), target_names=iris.target_names))

---
## 6. Optimiser et comparer

In [ ]:
# Meilleur K pour le KNN
scores_k = {}
for k in range(1, 21):
    knn_k = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    scores_k[k] = accuracy_score(y_test, knn_k.predict(X_test_s))

plt.plot(list(scores_k.keys()), list(scores_k.values()), marker="o")
plt.xlabel("K"); plt.ylabel("Précision"); plt.title("Précision du KNN selon K")
plt.grid(alpha=0.3); plt.show()

print(f"Meilleur K : {max(scores_k, key=scores_k.get)}")

In [ ]:
# Validation croisée (plus robuste)
cv = cross_val_score(arbre, X, y, cv=5)
print(f"Validation croisée (arbre, 5-fold) : {cv.mean():.3f} ± {cv.std():.3f}")

In [ ]:
# Visualiser l'arbre
plt.figure(figsize=(16, 8))
plot_tree(arbre, filled=True, feature_names=iris.feature_names,
          class_names=iris.target_names, rounded=True, fontsize=10)
plt.title("Arbre de décision — Iris")
plt.show()

---
## 7. 🏆 Extension — Prédire une nouvelle fleur

In [ ]:
def predire_espece(sepal_l, sepal_w, petal_l, petal_w, modele=arbre):
    """Prédit l'espèce d'une fleur à partir de ses 4 mesures."""
    mesures = [[sepal_l, sepal_w, petal_l, petal_w]]
    return iris.target_names[modele.predict(mesures)[0]]

print("Petits pétales →", predire_espece(5.1, 3.5, 1.4, 0.2))   # setosa
print("Grands pétales →", predire_espece(6.7, 3.0, 5.2, 2.3))   # virginica

---
## 🎉 Checkpoint terminé !

Vous avez réalisé un projet ML complet : exploration → visualisation → préparation →
entraînement (3 modèles) → évaluation (matrice de confusion, rapport) → optimisation.

*📘 Module Data Science — Checkpoint 5 : Iris | Bootcamp Data Science*